# FEDE Embedding Fine-Tuning Pipeline

Run each stage in order. Each cell is independent — you can re-run any stage without repeating earlier ones.

| Stage | What it does |
|---|---|
| 0 | Sanity checks 
| 1 | Build training dataset | 
| 2 | Generate eval queries | 
| 3 | Train Round 1 |
| 4 | Mine hard negatives | 
| 5 | Train Round 2 | 
| 6 | Evaluate | 
| 7 | Index corpus into Qdrant |

## Configuration

Edit these before running anything.

In [1]:
# ── How many movies to use for training (rest become eval pool) ──────────────
TRAIN_MOVIES     = 1200
N_EVAL_QUERIES   = 100       # held-out eval queries to generate

# ── Output directories ───────────────────────────────────────────────────────
ROUND1_MODEL_DIR = "fede-embeddinggemma/round1"
ROUND2_MODEL_DIR = "fede-embeddinggemma/round2"

# ── Training overrides (None = use config.py defaults) ───────────────────────
EPOCHS      = None   # e.g. 1 for a quick test
BATCH_SIZE  = None   # e.g. 8 if you hit OOM during training
LR          = None   # e.g. 1e-5
USE_LORA    = False  # set True if full fine-tune OOMs

# ── Evaluation ───────────────────────────────────────────────────────────────
EVAL_MOVIES = None   # None = full 1338-movie corpus; set e.g. 500 to save RAM

---
## Stage 0 — Sanity checks

Verify environment, credentials, and data before committing hours of compute.

In [2]:
import logging, os, json
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("pipeline")

# ── Project root ─────────────────────────────────────────────────────────────
import finetuning.config as cfg
log.info("Project root: %s", cfg.PROJECT_ROOT)

# ── Tagged scripts ────────────────────────────────────────────────────────────
tagged_files = list(cfg.TAGGED_SCRIPTS_DIR.glob("*.txt"))
log.info("Tagged scripts found: %d", len(tagged_files))
assert len(tagged_files) > 0, f"No tagged scripts in {cfg.TAGGED_SCRIPTS_DIR}"

# ── Metadata ─────────────────────────────────────────────────────────────────
with open(cfg.METADATA_PATH) as f:
    meta = json.load(f)
log.info("Metadata entries: %d", len(meta))

# ── Gemini API key (used by Batch API for dataset generation) ─────────────────
gemini_key = os.getenv("GEMINI_API_KEY", "")
assert gemini_key, "GEMINI_API_KEY is not set in .env"
log.info("Gemini key: %s…%s", gemini_key[:8], gemini_key[-4:])

# ── HF token (for gated embedding model) ─────────────────────────────────────
hf_token = os.getenv("HF_TOKEN", "")
if hf_token:
    log.info("HF_TOKEN present")
else:
    log.warning("HF_TOKEN not set — fine unless you switched to a gated model")

# ── Existing data ─────────────────────────────────────────────────────────────
raw_path  = cfg.FINETUNING_DATA_DIR / "raw_queries.jsonl"
r1_path   = cfg.FINETUNING_DATA_DIR / "training_pairs_r1.jsonl"
r2_path   = cfg.FINETUNING_DATA_DIR / "training_pairs_r2.jsonl"
eval_path = cfg.FINETUNING_DATA_DIR / "eval_queries.json"
ckpt_path = cfg.FINETUNING_DATA_DIR / "querygen_checkpoint.json"

def _count(p): return sum(1 for _ in open(p)) if p.exists() else 0

print(f"\nCurrent data state:")
print(f"  raw_queries.jsonl       : {_count(raw_path):>6} lines")
print(f"  training_pairs_r1.jsonl : {_count(r1_path):>6} lines")
print(f"  training_pairs_r2.jsonl : {_count(r2_path):>6} lines")
print(f"  eval_queries.json       : {'exists' if eval_path.exists() else 'missing'}")
print(f"  querygen_checkpoint.json: {'exists' if ckpt_path.exists() else 'missing'}")
if ckpt_path.exists():
    ckpt = json.loads(ckpt_path.read_text())
    print(f"    → {len(ckpt.get('processed_movies', []))} movies in checkpoint, status={ckpt.get('status', 'in_progress')}")

print("\n✅ Sanity checks passed")

2026-03-22 07:24:12,794 [INFO] Project root: /Users/anze/NLP/fede
2026-03-22 07:24:12,802 [INFO] Tagged scripts found: 1338
2026-03-22 07:24:12,811 [INFO] Metadata entries: 1384
2026-03-22 07:24:12,811 [INFO] Gemini key: AIzaSyC1…QQDE
2026-03-22 07:24:12,812 [INFO] HF_TOKEN present



Current data state:
  training_pairs_r1.jsonl :      6 lines
  training_pairs_r2.jsonl :      0 lines
  eval_queries.json       : missing
  querygen_checkpoint.json: missing

✅ Sanity checks passed


---
## Stage 1 — Build training dataset (Gemini Batch API)

Uses the Gemini Batch API to generate all synthetic queries in a single batch job (50% cheaper, no rate limits).

Three sub-steps:
1. **1a** — Build prompt JSONL + submit batch job (~30 seconds)
2. **1b** — Poll until the job completes (~5-40 minutes)
3. **1c** — Download results + post-process into training pairs

Save the `job_name` printed in 1a — you can re-run 1b/1c independently (e.g. after a kernel restart).

In [4]:
# ── Stage 1a: Build prompt JSONL + submit batch job ──────────────────────────
from finetuning.dataset.batch_builder import BatchDatasetBuilder

builder = BatchDatasetBuilder(max_movies=TRAIN_MOVIES)
job_name = builder.build_and_submit()

print(f"\n✅ Stage 1a complete — batch job submitted: {job_name}")
print(f"Save this job_name in case you need to re-run 1b/1c after a kernel restart.")

2026-03-21 17:33:30,969 [INFO] Scene corpus built: 1200 movies, 117100 total scenes
2026-03-21 17:33:31,719 [INFO] Batch JSONL built: 1178 synopsis + 3576 scene prompts → /Users/anze/NLP/fede/data/finetuning/batch_requests.jsonl
2026-03-21 17:33:31,719 [INFO] Uploading /Users/anze/NLP/fede/data/finetuning/batch_requests.jsonl to Gemini Files API …
2026-03-21 17:33:32,374 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
2026-03-21 17:33:33,044 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWwkUIqChesjLVpWfrOaqJmNXAmv3qRNw9cwwSTNXKFaSiIxY8tEejeACd4p8BjONJW5BA1Q46KAlTgDxJ1ziZyFPRB8pc0zlPso-9hYmmI&upload_protocol=resumable "HTTP/1.1 200 OK"
2026-03-21 17:33:33,578 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWwkUIqChesjLVpWfrOaqJmNXAmv3qRNw9cwwSTNXKFaSiIxY8tEejeACd4p8BjONJW5BA1Q46KAlTgDxJ1ziZyFPRB8pc0zlPso-9hYmmI&upload_p

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [5]:
builder._model = "gemini-2.0-flash"
job_name = builder.submit_batch_job()
print(f"Batch job submitted: {job_name}")

2026-03-21 17:37:36,406 [INFO] Uploading /Users/anze/NLP/fede/data/finetuning/batch_requests.jsonl to Gemini Files API …
2026-03-21 17:37:36,935 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
2026-03-21 17:37:38,214 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWzFWGfYomvKc96pQD_tQ2zAwkEOuGo4aFfmoEaavrAj9yNwlpwDkVHdo1TYtZuHzvTVSZHE4PfxiEkWtbbrOIIP1LDeiPNpx4tAvnbs4A&upload_protocol=resumable "HTTP/1.1 200 OK"
2026-03-21 17:37:39,063 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWzFWGfYomvKc96pQD_tQ2zAwkEOuGo4aFfmoEaavrAj9yNwlpwDkVHdo1TYtZuHzvTVSZHE4PfxiEkWtbbrOIIP1LDeiPNpx4tAvnbs4A&upload_protocol=resumable "HTTP/1.1 200 OK"
2026-03-21 17:37:39,741 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWzFWGfYomvKc96pQD_tQ2zAwkEOuGo4aFfmoEaavrAj9yNwlpwDkVHdo1TYtZuHzvTVS

Batch job submitted: batches/0597gpykw6qpr7ynen2w98o5qhetnmbt82r2


In [ ]:
# ── Stage 1b: Poll until batch job completes ─────────────────────────────────
# If you restarted the kernel, uncomment and set the job_name:
# job_name = "batches/XXXXXXX"

# Recreate builder if needed (e.g. after kernel restart)
if "builder" not in dir() or builder is None:
    from finetuning.dataset.batch_builder import BatchDatasetBuilder
    builder = BatchDatasetBuilder(max_movies=TRAIN_MOVIES)

state = builder.poll_until_done(job_name)
print(f"\n✅ Stage 1b complete — job state: {state}")

In [5]:
# ── Stage 1c: Download results + post-process into training pairs ────────────
# Recreate builder if needed (e.g. after kernel restart)
if "builder" not in dir() or builder is None:
    from finetuning.dataset.batch_builder import BatchDatasetBuilder
    builder = BatchDatasetBuilder(max_movies=TRAIN_MOVIES)

r1_path = builder.process_results(job_name=job_name)
pairs = _count(r1_path)
print(f"\n✅ Stage 1 complete: {pairs} training pairs → {r1_path}")

2026-03-21 17:27:48,640 [INFO] HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/batches/p0e2a64dk839jkgl0ytamuc2wlei2yb4s752 "HTTP/1.1 200 OK"
2026-03-21 17:27:48,652 [INFO] Downloading result file files/batch-p0e2a64dk839jkgl0ytamuc2wlei2yb4s752 …
2026-03-21 17:27:48,685 [INFO] HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/files/batch-p0e2a64dk839jkgl0ytamuc2wlei2yb4s752:download?alt=media "HTTP/1.1 302 Found"
2026-03-21 17:27:49,429 [INFO] HTTP Request: GET https://generativelanguage.googleapis.com/download/v1beta/files/batch-p0e2a64dk839jkgl0ytamuc2wlei2yb4s752:download?alt=media "HTTP/1.1 200 OK"
2026-03-21 17:27:49,536 [INFO] Results downloaded: 4754 lines → /Users/anze/NLP/fede/data/finetuning/batch_results.jsonl
2026-03-21 17:27:49,549 [INFO] Loading embedding model for positive assignment …
2026-03-21 17:27:49,550 [INFO] Loading embedding model: google/embeddinggemma-300m
2026-03-21 17:27:49,562 [INFO] Load pretrained SentenceTransformer: 


✅ Stage 1 complete: 0 training pairs → /Users/anze/NLP/fede/data/finetuning/training_pairs_r1.jsonl


In [11]:
# Clear the stale batch checkpoint
import os
ckpt = cfg.FINETUNING_DATA_DIR / "querygen_checkpoint.json"
if ckpt.exists():
    os.remove(ckpt)
    print("Checkpoint cleared")

Checkpoint cleared


In [ ]:
from finetuning.dataset.dataset_builder import DatasetBuilder

builder = DatasetBuilder(max_movies=TRAIN_MOVIES)
r1_path = builder.build(resume=True)

pairs = _count(r1_path)
print(f"\n✅ Stage 1 complete: {pairs} pairs → {r1_path}")

/Users/anze/NLP/fede/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-22 07:24:31,727 [INFO] Scene corpus built: 1200 movies, 117100 total scenes
2026-03-22 07:24:31,732 [INFO] Found 1 extra movies in JSONL not in checkpoint — adding to skip set
2026-03-22 07:24:31,733 [INFO] 1199 movies to process (1 already done)
2026-03-22 07:24:31,733 [INFO] [2/1200] Generating queries for Freddy vs. Jason
2026-03-22 07:24:31,740 [INFO] Loading embedding model google/embeddinggemma-300m on mps …
2026-03-22 07:24:31,741 [INFO] Loading embedding model: google/embeddinggemma-300m
2026-03-22 07:24:31,744 [INFO] Load pretrained SentenceTransformer: google/embeddinggemma-300m
2026-03-22 07:24:31,958 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1

In [13]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Write one short search query about a movie scene with a car chase.",
)
print(response.text)

2026-03-22 07:20:33,232 [INFO] AFC is enabled with max remote calls: 10.
2026-03-22 07:20:39,221 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


**movie car chase scene**


In [14]:
from finetuning.dataset.query_generator import QueryGenerator

qgen = QueryGenerator()
queries = qgen.generate_synopsis_queries(
    overview="A young woman discovers she has superpowers and must save her city from a mysterious villain.",
    movie_title="Test Movie",
    n=4,
)
print(queries)

2026-03-22 07:20:54,570 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


['A scene where someone realizes they have extraordinary abilities.', 'Scenes showing a hero defending their city from a powerful unknown enemy.', 'A young person learning to use newly acquired special powers.', 'Moments where a city is in danger and needs a new protector.']


---
## Stage 2 — Generate eval queries

Generates ~100 held-out evaluation queries from movies **not** in the training set.

⚠️ Must run **after Stage 1** — reads the checkpoint to know which movies to exclude.

In [ ]:
from finetuning.corpus.scene_corpus import load_metadata
from finetuning.dataset.query_generator import load_checkpoint
from finetuning.evaluation.dataset_generator import generate_eval_dataset

ckpt = load_checkpoint()
assert ckpt, "No checkpoint found — run Stage 1 first"
training_ids = set(ckpt.get("processed_movies", []))
log.info("Excluding %d training movies from eval pool", len(training_ids))

metadata = load_metadata()
eval_path = generate_eval_dataset(
    corpus_movie_ids=training_ids,
    metadata=metadata,
    n=N_EVAL_QUERIES,
)

with open(eval_path) as f:
    eval_queries = json.load(f)
print(f"\n✅ Stage 2 complete: {len(eval_queries)} eval queries → {eval_path}")
print("Sample:", eval_queries[0])

In [ ]:
import gc

# Free Stage 1 objects before loading the training model.
# builder holds the embedding model + full corpus in RAM — ~2–3 GB.
# Releasing them prevents OOM when the training model loads on top.
for _var in ("builder",):
    if _var in dir():
        del globals()[_var]
gc.collect()
print("🧹 Stage 1 objects released")

---
## Stage 3 — Train Round 1

Fine-tunes the base embedding model on the round-1 dataset using `CachedMultipleNegativesRankingLoss`.
Runs retrieval eval (MRR / Accuracy@k) after each epoch if `eval_queries.json` exists.

💡 If you OOM here, set `BATCH_SIZE = 8` or `USE_LORA = True` in the config cell above.

In [ ]:
from finetuning.training.model import load_model
from finetuning.training.trainer import build_evaluator, build_trainer, load_training_dataset
from finetuning.corpus.scene_corpus import build_scene_corpus

model      = load_model()  # base model from config
train_data = load_training_dataset(r1_path)
log.info("Training samples: %d", len(train_data))

evaluator = None
if eval_path.exists():
    log.info("Building eval corpus for per-epoch evaluation …")
    eval_corpus = build_scene_corpus(max_movies=EVAL_MOVIES)
    evaluator = build_evaluator(eval_path, eval_corpus)
else:
    log.warning("eval_queries.json not found — training without per-epoch eval")

train_kwargs = {}
if EPOCHS     is not None: train_kwargs["num_epochs"]    = EPOCHS
if BATCH_SIZE is not None: train_kwargs["batch_size"]    = BATCH_SIZE
if LR         is not None: train_kwargs["learning_rate"] = LR

trainer = build_trainer(
    model=model,
    train_dataset=train_data,
    output_dir=ROUND1_MODEL_DIR,
    evaluator=evaluator,
    use_lora=USE_LORA,
    **train_kwargs,
)

trainer.train()
model.save(ROUND1_MODEL_DIR)
print(f"\n✅ Stage 3 complete: round-1 model saved to {ROUND1_MODEL_DIR}")

In [ ]:
import gc

# Free training objects before loading the corpus + model for hard-negative mining.
for _var in ("model", "trainer", "train_data", "eval_corpus", "evaluator"):
    if _var in globals():
        del globals()[_var]
gc.collect()
print("🧹 Stage 3 objects released")

---
## Stage 4 — Mine hard negatives

Uses the round-1 model to replace random negatives with semantically close (but wrong) scenes.
This makes round-2 training significantly harder and usually gives the biggest quality jump.

⚠️ Peak RAM: loads round-1 model + full scene corpus simultaneously.
Pass `--movies N` equivalent by setting `EVAL_MOVIES` above if you OOM.

In [ ]:
from finetuning.dataset.negative_miner import CorpusIndex, mine_hard_negatives

r1_model = load_model(ROUND1_MODEL_DIR)

log.info("Building scene corpus …")
corpus = build_scene_corpus(max_movies=EVAL_MOVIES)

log.info("Building embedding index …")
index = CorpusIndex.build(corpus, r1_model)

log.info("Loading round-1 dataset …")
with open(r1_path) as f:
    r1_rows = [json.loads(line) for line in f]

log.info("Mining hard negatives for %d pairs …", len(r1_rows))
r2_path.parent.mkdir(parents=True, exist_ok=True)
with open(r2_path, "w", encoding="utf-8") as out:
    for i, row in enumerate(r1_rows, 1):
        hard_negs = mine_hard_negatives(
            query=row["anchor"],
            movie_id=row["movie_id"],
            index=index,
            model=r1_model,
            n=cfg.HARD_NEGATIVES_PER_QUERY,
        )
        row["negatives"] = hard_negs
        out.write(json.dumps(row, ensure_ascii=False) + "\n")
        if i % 500 == 0:
            log.info("  %d / %d", i, len(r1_rows))

print(f"\n✅ Stage 4 complete: {len(r1_rows)} pairs with hard negatives → {r2_path}")

---
## Stage 5 — Train Round 2

Continues fine-tuning from the round-1 checkpoint using the hard-negative dataset.

In [ ]:
r2_model   = load_model(ROUND1_MODEL_DIR)  # start from round-1 weights
train_data2 = load_training_dataset(r2_path)
log.info("Round-2 training samples: %d", len(train_data2))

evaluator2 = None
if eval_path.exists():
    eval_corpus2 = build_scene_corpus(max_movies=EVAL_MOVIES)
    evaluator2 = build_evaluator(eval_path, eval_corpus2)

trainer2 = build_trainer(
    model=r2_model,
    train_dataset=train_data2,
    output_dir=ROUND2_MODEL_DIR,
    evaluator=evaluator2,
    use_lora=USE_LORA,
    **train_kwargs,
)

trainer2.train()
r2_model.save(ROUND2_MODEL_DIR)
print(f"\n✅ Stage 5 complete: round-2 model saved to {ROUND2_MODEL_DIR}")

In [ ]:
import gc

# Free the corpus index + r1_model before loading the training model for Round 2.
# The CorpusIndex holds all 117k scene embeddings in RAM — ~350 MB.
for _var in ("r1_model", "index", "corpus", "r1_rows"):
    if _var in globals():
        del globals()[_var]
gc.collect()
print("🧹 Stage 4 objects released")

---
## Stage 6 — Evaluate

Runs the held-out eval queries against the full scene corpus and prints MRR / Accuracy@k.

In [ ]:
from finetuning.evaluation.pipeline import run_pipeline
from finetuning.evaluation.semantic_retriever import SemanticRetriever

# Evaluate base model for comparison
models_to_eval = {
    "base"   : None,              # None → loads config.EMBEDDING_MODEL_ID
    "round1" : ROUND1_MODEL_DIR,
    "round2" : ROUND2_MODEL_DIR,
}

results = {}
for name, model_path in models_to_eval.items():
    if model_path is not None and not Path(model_path).exists():
        log.warning("Skipping %s — directory not found", name)
        continue
    log.info("Evaluating %s …", name)
    m = load_model(model_path)
    retriever = SemanticRetriever(m)
    metrics = run_pipeline(retriever, eval_path=eval_path)
    results[name] = metrics["summary"]

# Pretty-print comparison table
print(f"\n{'Model':<10} {'MRR':>6}  {'Acc@5':>6}  {'Acc@10':>7}  {'Acc@20':>7}")
print("-" * 45)
for name, s in results.items():
    print(
        f"{name:<10} {s.get('mrr', 0):>6.3f}  "
        f"{s.get('accuracy_at_5', 0):>6.3f}  "
        f"{s.get('accuracy_at_10', 0):>7.3f}  "
        f"{s.get('accuracy_at_20', 0):>7.3f}"
    )

# Save report
report_path = cfg.FINETUNING_DATA_DIR / "eval_report.json"
report_path.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"\n✅ Stage 6 complete — report saved to {report_path}")

---
## Stage 7 — Index corpus into Qdrant

Embeds all scenes with the final model and upserts them into Qdrant.

⚠️ Make sure Qdrant is running before executing this cell.

In [ ]:
from finetuning.training.model import encode_documents
from vector_db.indexer import SceneRecord, ScriptIndexer

final_model = load_model(ROUND2_MODEL_DIR)
corpus      = build_scene_corpus()  # full corpus, no max_movies
indexer     = ScriptIndexer()

total_scenes = sum(len(e.scenes) for e in corpus.values())
log.info("Indexing %d movies, %d scenes …", len(corpus), total_scenes)

indexed = 0
for i, (movie_id, entry) in enumerate(corpus.items(), 1):
    scene_texts = [s.text for s in entry.scenes]
    embeddings  = encode_documents(final_model, scene_texts, batch_size=64)

    records = [
        SceneRecord(
            movie_id=scene.movie_id,
            movie_title=scene.movie_title,
            scene_id=scene.scene_id,
            scene_index=scene.scene_index,
            text=scene.text,
            embedding=emb.tolist(),
            scene_title=scene.scene_title,
            character_names=scene.character_names,
        )
        for scene, emb in zip(entry.scenes, embeddings)
    ]

    indexer.index_movie_batch(scenes=records, sentences=[])
    indexed += len(records)

    if i % 100 == 0:
        log.info("  %d / %d movies indexed (%d scenes)", i, len(corpus), indexed)

print(f"\n✅ Stage 7 complete: {len(corpus)} movies, {indexed} scenes indexed")